# Fine-Tuning CampusFlow AI Coach (`Qwen2.5-7B-Instruct` QLoRA -> GGUF Ollama)

**Dataset yang dibutuhkan (dari `backend/scripts/`):**
- `balanced_dataset_narratives_train_messages.jsonl` (`462` baris)
- `balanced_dataset_narratives_val_messages.jsonl` (`52` baris)

**Pastikan Runtime Colab memakai GPU minimal 15 GB VRAM (`Runtime` -> `Change runtime type` -> `T4 GPU`).**

In [ ]:
!nvidia-smi
!pip install -q unsloth

In [ ]:
import os
import glob

TRAIN_NAME = "balanced_dataset_narratives_train_messages.jsonl"
VAL_NAME = "balanced_dataset_narratives_val_messages.jsonl"

# Opsi A: cari di /content atau Google Drive jika sudah di-mount
def find_file(filename):
    for pattern in [f"/content/{filename}", f"/content/drive/MyDrive/**/{filename}"]:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            return matches[0]
    return None

# Jika file disimpan di Google Drive, uncomment 2 baris berikut:
# from google.colab import drive
# drive.mount('/content/drive')

train_path = find_file(TRAIN_NAME)
val_path = find_file(VAL_NAME)

if not train_path or not val_path:
    from google.colab import files
    print(f"Silakan upload {TRAIN_NAME} dan {VAL_NAME}...")
    uploaded = files.upload()
    train_path = find_file(TRAIN_NAME)
    val_path = find_file(VAL_NAME)

assert train_path and os.path.exists(train_path), f"{TRAIN_NAME} tidak ditemukan!"
assert val_path and os.path.exists(val_path), f"{VAL_NAME} tidak ditemukan!"
print("Train:", train_path, "| baris:", sum(1 for _ in open(train_path, encoding="utf-8")))
print("Val  :", val_path, "| baris:", sum(1 for _ in open(val_path, encoding="utf-8")))

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch

MAX_SEQ_LENGTH = 2048
BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
print("Model & LoRA siap di:", torch.cuda.get_device_name(0))

In [ ]:
import json
from datasets import load_dataset

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

train_ds = load_dataset("json", data_files=train_path, split="train").shuffle(seed=42).map(
    formatting_prompts_func, batched=True
)
val_ds = load_dataset("json", data_files=val_path, split="train").map(
    formatting_prompts_func, batched=True
)

with open(train_path, encoding="utf-8") as f:
    TRAINING_SYSTEM_PROMPT = json.loads(f.readline())["messages"][0]["content"]

print("Train rows:", len(train_ds), "| Val rows:", len(val_ds))
print("Contoh 400 karakter pertama:\n", train_ds[0]["text"][:400])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=30,
        save_strategy="steps",
        save_steps=30,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs_campusflow_7b",
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

stats = trainer.train()
print(stats)

In [ ]:
# Smoke test inferensi sebelum export GGUF
FastLanguageModel.for_inference(model)

def ask_local(user_json_str):
    msgs = [
        {"role": "system", "content": TRAINING_SYSTEM_PROMPT},
        {"role": "user", "content": user_json_str},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=320, use_cache=True, do_sample=False)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

with open(val_path, encoding="utf-8") as f:
    val_rows = [json.loads(line) for line in f if line.strip()]

print("=== Uji Baris Val #1 (Wellbeing) ===")
print(ask_local(val_rows[0]["messages"][1]["content"]))
print("\n=== Uji Baris Val Terakhir (Out-of-Scope) ===")
print(ask_local(val_rows[-1]["messages"][1]["content"]))

In [ ]:
import glob
import os
import shutil

EXPORT_DIR = "campusflow_7b_gguf"
model.save_pretrained_gguf(EXPORT_DIR, tokenizer, quantization_method="q4_k_m")

gguf_files = glob.glob(f"{EXPORT_DIR}/*.gguf")
assert gguf_files, "File GGUF tidak ditemukan setelah export!"
gguf_basename = os.path.basename(gguf_files[0])

# Tulis Modelfile dengan temperature 0 dan SYSTEM prompt yang konsisten dengan training + ollamaCoach.ts
modelfile_template = '''FROM ./__GGUF_FILE__
PARAMETER temperature 0
PARAMETER top_p 1
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 4096
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"

TEMPLATE """{{- if .Messages }}
{{- if or .System .Tools }}<|im_start|>system
{{- if .System }}
{{ .System }}
{{- end }}<|im_end|>
{{ end }}
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 -}}
{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
{{ if .Content }}{{ .Content }}{{ end }}{{ if not $last }}<|im_end|>
{{ end }}
{{ end }}
{{- if and (ne .Role "assistant") $last }}<|im_start|>assistant
{{ end }}
{{- end }}
{{- else }}
{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ end }}{{ .Response }}{{ if .Response }}<|im_end|>{{ end }}"""

SYSTEM """__SYSTEM_PROMPT__"""
'''
modelfile_content = modelfile_template.replace("__GGUF_FILE__", gguf_basename).replace(
    "__SYSTEM_PROMPT__", TRAINING_SYSTEM_PROMPT.replace('"""', '\"\"\"')
)

modelfile_path = os.path.join(EXPORT_DIR, "Modelfile")
with open(modelfile_path, "w", encoding="utf-8") as f:
    f.write(modelfile_content)

print("Siap! File GGUF:", gguf_files[0])
print("Siap! Modelfile:", modelfile_path)

# Jika Google Drive ter-mount, salin otomatis agar mudah didownload:
if os.path.exists("/content/drive/MyDrive"):
    drive_out = "/content/drive/MyDrive/campusflow_7b_gguf"
    os.makedirs(drive_out, exist_ok=True)
    shutil.copy2(gguf_files[0], os.path.join(drive_out, gguf_basename))
    shutil.copy2(modelfile_path, os.path.join(drive_out, "Modelfile"))
    print("Disalin ke Google Drive:", drive_out)